# Car Data Project: Model Evaluation
This notebook calculates the foundational metrics for the car price prediction model, including dataset sizes and regression performance scores (R², MAE, RMSE).

In [23]:
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings('ignore')

## 1. Dataset Overview
Checking the total row counts for the raw data and the test split.

In [24]:
try:
    # Notice the ../ to go up one directory from the notebooks folder
    df_raw = pd.read_csv('../data/used_cars_UK.csv')
    print(f"Raw Dataset Row Count: {len(df_raw):,}")
    
    df_test = pd.read_csv('../data/test_data.csv')
    print(f"Test Dataset Row Count: {len(df_test):,}")
except FileNotFoundError as e:
    print(f"File not found: {e.filename}. Make sure you are running this from the notebooks/ directory.")

Raw Dataset Row Count: 3,685
Test Dataset Row Count: 3


## 2. Model Performance Metrics
Loading the Scikit-learn pipeline and evaluating its predictions against the unseen test data.

In [25]:
test_data = pd.read_csv('../data/test_data.csv')
print(test_data.columns.tolist())

['Make', 'Model', 'Mileage', 'Engine_Size', 'Price']


In [26]:
import sqlite3
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

try:
    # 1. Connect to the processed SQLite database
    conn = sqlite3.connect('../database/car_data.db')
    
    # Dynamically find the main table name
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
    table_name = tables['name'].iloc[0]
    
    # Load the real dataset
    real_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    print(f"Successfully loaded {len(real_data):,} real records from the database.\n")
    
    # FIX: Rename 'engine_size' to 'displacement' to match the model's training
    real_data = real_data.rename(columns={'engine_size': 'displacement'})
    
    # 2. Prepare the data
    model = joblib.load('../models/final_model.joblib')
    expected_features = ['make', 'model', 'mileage', 'displacement']
    
    # Target column check
    target_col = 'price' if 'price' in real_data.columns else 'Price'
    
    # Drop any rows with missing values in our crucial columns
    eval_data = real_data.dropna(subset=expected_features + [target_col])
    
    X_real = eval_data[expected_features]
    y_real = eval_data[target_col]
    
    # 3. Predict and Evaluate
    y_pred = model.predict(X_real)
    
    r2 = r2_score(y_real, y_pred)
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    
    print("--- REAL Model Performance ---")
    print(f"R² Score: {r2:.4f}")
    print(f"Mean Absolute Error (MAE): £{mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): £{rmse:,.2f}")
    
except Exception as e:
    print(f"Error evaluating on database: {e}")

Successfully loaded 3,640 real records from the database.

--- REAL Model Performance ---
R² Score: 0.4354
Mean Absolute Error (MAE): £2,507.70
Root Mean Squared Error (RMSE): £3,381.72
